# TF-IDF baseline

Do the same fairness check but with TF-IDF instead of sbert. Mostly to see if sbert is doing something different from just counting words.

In [ ]:
!pip install scikit-learn pandas

In [ ]:
import os, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
up = files.upload()
for f in up:
    os.rename(f, f"data/{f}")

In [ ]:
jobs = pd.read_csv("data/jobs.csv")
res = pd.read_csv("data/resume_variants.csv")
jobs["job_text"] = jobs["title"]+" "+jobs["domain"]+" "+jobs["company_name"]+" "+jobs["job_description"]

In [ ]:
# fit one vectorizer on everything so the vocab matches
vec = TfidfVectorizer(stop_words="english", lowercase=True)
vec.fit(list(res["resume_text"]) + list(jobs["job_text"]))
rv = vec.transform(res["resume_text"])
jv = vec.transform(jobs["job_text"])
print(rv.shape, jv.shape)

In [ ]:
out = []
for i in range(len(res)):
    for j in range(len(jobs)):
        s = float(cosine_similarity(rv[i], jv[j])[0][0])
        out.append({
            "resume_id": res.iloc[i]["resume_id"],
            "version": res.iloc[i]["version"],
            "changed_signal": res.iloc[i]["changed_signal"],
            "job_id": jobs.iloc[j]["job_id"],
            "job_title": jobs.iloc[j]["title"],
            "tfidf_similarity_score": s,
        })
scores = pd.DataFrame(out)

In [ ]:
orig = scores[scores.version=="original"][["resume_id","job_id","job_title","tfidf_similarity_score"]]
orig = orig.rename(columns={"tfidf_similarity_score":"tfidf_original_score"})
ch = scores[scores.version!="original"].rename(columns={"tfidf_similarity_score":"tfidf_changed_score"})
cmp = ch.merge(orig, on=["resume_id","job_id","job_title"], how="left")
cmp["tfidf_score_difference"] = cmp["tfidf_changed_score"] - cmp["tfidf_original_score"]
cmp["tfidf_absolute_difference"] = cmp["tfidf_score_difference"].abs()
summary = cmp.groupby("changed_signal").agg(
    average_tfidf_score_difference=("tfidf_score_difference","mean"),
    average_tfidf_absolute_difference=("tfidf_absolute_difference","mean"),
    max_tfidf_absolute_difference=("tfidf_absolute_difference","max"),
).reset_index()
summary

TF-IDF moves a lot less than sbert. So whatever sbert is doing with these demographic words, it isn't just word-overlap.

In [ ]:
scores.to_csv("results/tfidf_scores.csv", index=False)
cmp.to_csv("results/tfidf_comparison.csv", index=False)
summary.to_csv("results/tfidf_summary.csv", index=False)
for f in ["tfidf_scores.csv","tfidf_comparison.csv","tfidf_summary.csv"]:
    files.download(f"results/{f}")